1. Install and Import Required Libraries

In [1]:
!pip install contractions emoji beautifulsoup4 -q


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import re
import contractions
import emoji
import nltk

from bs4 import BeautifulSoup
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download required resources
nltk.download("punkt")
nltk.download('punkt_tab')
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gopin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\gopin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\gopin\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\gopin\AppData\Roaming\nltk_data...


True

2. Mount Google Drive and Load the Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [5]:
train = pd.read_csv(r"C:\Users\gopin\OneDrive\Documents\Data_Science\Streamlit Training files\Comment Toxicity\ctm\Scripts\data\train.csv")
test = pd.read_csv(r"C:\Users\gopin\OneDrive\Documents\Data_Science\Streamlit Training files\Comment Toxicity\ctm\Scripts\data\test.csv")


print(train.shape)
print(test.shape)

train.head()

(159571, 8)
(153164, 2)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [6]:
print(train.isnull().sum())

train["comment_text"] = train["comment_text"].fillna("")
test["comment_text"] = test["comment_text"].fillna("")

id               0
comment_text     0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
dtype: int64


3. Clean and Preprocess Text Data

In [7]:
lemmatizer = WordNetLemmatizer()

In [8]:
def clean_text(text):

    text = str(text)

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove emails
    text = re.sub(r"\S+@\S+", " ", text)

    # Now remove HTML
    text = BeautifulSoup(text, "html.parser").get_text()

    # Remove emojis
    text = emoji.replace_emoji(text, replace=" ")

    # Expand contractions
    text = contractions.fix(text)

    # Keep alphabets, ! and ?
    text = re.sub(r"[^a-z!?\\s]", " ", text)
    # Convert !!! -> !
    # Convert ???? -> ?
    text = re.sub(r"!+", " ! ", text)
    text = re.sub(r"\?+", " ? ", text)
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenize
    words = word_tokenize(text)

    # Lemmatize
    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

In [9]:
train["clean_text"] = train["comment_text"].apply(clean_text)

test["clean_text"] = test["comment_text"].apply(clean_text)

print("Text Cleaning Done")

Text Cleaning Done


4. Validate and Filter Cleaned Text Data

In [10]:
for i in range(5):

    print("="*80)

    print("Original:\n")
    print(train.loc[i,"comment_text"])

    print("\nCleaned:\n")
    print(train.loc[i,"clean_text"])

    print("="*80)

Original:

Explanation
Why the edits made under my username Hardcore Metallica Fan were reverted? They weren't vandalisms, just closure on some GAs after I voted at New York Dolls FAC. And please don't remove the template from the talk page since I'm retired now.89.205.38.27

Cleaned:

explanation why the edits made under my username hardcore metallica fan were reverted ? they were not vandalism just closure on some gas after i voted at new york doll fac and please do not remove the template from the talk page since i am retired now
Original:

D'aww! He matches this background colour I'm seemingly stuck with. Thanks.  (talk) 21:51, January 11, 2016 (UTC)

Cleaned:

d aww ! he match this background colour i am seemingly stuck with thanks talk january utc
Original:

Hey man, I'm really not trying to edit war. It's just that this guy is constantly removing relevant information and talking to me through edits instead of my talk page. He seems to care more about the formatting than the actu

In [11]:
empty_comments = (train["clean_text"].str.strip() == "").sum()

print("Empty comments:", empty_comments)

Empty comments: 24


In [12]:
train = train[train["clean_text"].str.strip() != ""]

In [13]:
train.shape

(159547, 9)

5. Analyze Word Count and Save Preprocessed Dataset

In [14]:
train["word_count"] = train["clean_text"].apply(lambda x: len(x.split()))

print(train["word_count"].describe())

count    159547.000000
mean         68.474218
std         101.511404
min           1.000000
25%          17.000000
50%          36.000000
75%          76.000000
max        1495.000000
Name: word_count, dtype: float64


In [15]:
train.to_csv("train_cleaned.csv", index=False)

test.to_csv("test_cleaned.csv", index=False)

print("Files Saved Successfully")

Files Saved Successfully


In [ ]:
from google.colab import files

files.download("train_cleaned.csv")
files.download("test_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>